# HW2: Building Explorer Agents for the Underwater Caldera

In this assignment, we will create agents that are designed to achieve different exploration tasks in the environment we have designed.

We will use different decision-making approaches and compare their performance.

Specifically, we will use active learning, planning, and reinforcement learning, and compare their performance on the task of finding the deepest point in the Caldera.


### Objectives:

The objective of HW2 is to explore different decision-making approaches and gain an understanding of how active learning, planning, and reinforcement learning can be used to guide an agent under limited information and resource constraints.

For this assignment, we extended our caldera environment. As before, it can be initialized with a set of pits, which define the caldera's topography, but now it can also be initialized with an externally supplied depth-map array.
The first figure below shows an example environment initialized with three Gaussian pits, while the following one shows an environment generated from an external depth map.


<!-- <img src="../images/envs/Environment 1.png" alt="Kolumbo submarine caldera" width="500" /> -->
![Kolumbo submarine caldera](../images/envs/Environment%201.png)


<!-- <img src="../images/envs/Environment 2.png" alt="Kolumbo submarine caldera" width="500" /> -->
![Kolumbo submarine caldera](../images/envs/Environment%202.png)



### Separating the Environment from the Agent and its Task ⚠️ 

A key design principle in this assignment is that the agent should not rely on full access to the environment internals when making decisions.

The agent may receive a reference to the environment because it needs to interact with it through the standard environment API, i.e., by executing actions and receiving observations. However, the decision-making logic should not directly inspect hidden environment state such as the full depth map, obstacle internals, or any other information that would not be available to the agent in a realistic setting.

Instead, each agent receives a `task_info` dictionary. This dictionary contains only the information that is allowed for the task. For example, it may include grid dimensions, movement size, energy costs, known vehicle locations, or other task-level parameters.



📌 Note that task_info can change if you realize you need specific information for your decision-making approach!

## General Guidelines

While we provide the complete framework for this assignment, the only files you will submit are:

- `to_implement.py` &mdash; where you implement the coding assignment (`select_candidate_queries`, `get_query_score`, `init_policy_fn`, `update_policy_fn`, `epsilon_update_fn`, `state_key_fn`, `step_update_fn`, `episode_update_fn`, `reward_function`, and the `LEARNING_PARAMS` dictionary).
- `answers.py` &mdash; where you answer the theoretical questions.

⚠️ **Warning**: these are the only files that will be examined &mdash; any changes you make to other files will be ignored by the grader. In particular, do not modify `caldera_env.py`, `agent.py`, `explorer.py`, `value_and_policy.py`, or `utils.py`.

# Setup

We load the framework code, define a handful of helpers that render the Caldera environment as RGB frames (so we can build short videos with `mediapy`), and create four reference environments that we will use for all sanity checks and visual tests in this notebook.

All plots are rendered to in-memory buffers, so the notebook also runs headlessly (e.g., under `xvfb-run` on a server).

In [ ]:
%load_ext autoreload
%autoreload 2

import io
import unittest

import matplotlib
import matplotlib.pyplot as plt

import mediapy as media
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import Markdown

from caldera_env import (
    CalderaEnv, DEFAULT_PIT_PARAMS, DEFAULT_PIT_WEIGHTS, ACTION_NAMES,
    MOVE_NORTH, MOVE_SOUTH, MOVE_EAST, MOVE_WEST, SAMPLE, NO_OP,
)
from explorer import ALExplorer, PlanningExplorer, RLExplorer, OBJECTIVES
import answers
import to_implement as ti

np.random.seed(0)


## Render & rollout helpers

Two small helpers let us turn the existing `env.visualize(...)` figure into video frames and run an agent for one episode while capturing a frame at every step. We use these throughout the notebook for the video sections.

In [ ]:
def render_env_frame(env, **visualize_kwargs):
    """Render the current environment state as an RGB array.

    Calls `env.visualize(...)`, dumps the matplotlib figure to an in-memory
    PNG buffer, decodes it back as a numpy array, and closes the figure so we
    don't accumulate handles when building long videos.
    """
    visualize_kwargs.setdefault('show_grid_lines', True)
    fig, _ = env.visualize(**visualize_kwargs)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=80)
    plt.close(fig)
    buf.seek(0)
    return np.array(Image.open(buf).convert('RGB'))


def rollout_with_frames(agent, env=None, max_frames=200, stop_sample_value_at_most=None):
    """Run one episode end-to-end, capturing a frame after every step.

    Functionally the same as `agent.run_episode()` but it also records the
    environment state into a list of RGB frames suitable for `media.show_video`.
    When `stop_sample_value_at_most` is provided, the rollout stops right after
    a fresh sample with value at or below that threshold. This is useful for
    visualizing an RL policy's ability to reach its best known sample without
    continuing to spend the remaining episode budget afterward.
    """
    if env is None:
        env = agent.env
    obs, info = env.reset()
    frames = [render_env_frame(env)]
    terminated = truncated = False
    trajectory = []
    while not (terminated or truncated or agent.is_done()) and len(frames) < max_frames:
        action = agent.select_action(obs, info)
        next_obs, reward, terminated, truncated, next_info = env.step(action)
        step_info = {
            'obs': obs, 'action': action, 'reward': float(reward),
            'next_obs': next_obs, 'terminated': bool(terminated),
            'truncated': bool(truncated), 'info': next_info,
        }
        agent.step_update(step_info)
        trajectory.append(step_info)
        frames.append(render_env_frame(env))

        stop_now = False
        if stop_sample_value_at_most is not None and action == SAMPLE:
            if int(next_obs.get('sampled_before', 1)) == 0:
                sampled_value = float(np.asarray(next_obs['value']).item())
                stop_now = sampled_value <= stop_sample_value_at_most
        if stop_now:
            break

        obs = next_obs
        info = next_info
    episode_info = {
        'trajectory': trajectory,
        'total_reward': float(sum(s['reward'] for s in trajectory)),
        'num_steps': len(trajectory),
    }
    agent.episode_update(episode_info)
    return frames, episode_info


## Reference environments

We create four environments of increasing difficulty:

- **Environment 1**: smooth depth field generated from three Gaussian pits, no obstacles. The simplest case.
- **Environment 2**: a 20x20 depth map drawn from a uniform random matrix (no pit structure at all). Tests that your solution handles externally supplied maps and cannot rely on pit structure.
- **Environment 3**: same depth field as Environment 1 but with several obstacle vehicles. Tests path planning and collision handling.
- **Environment 4**: same depth field as Environment 1 with many obstacle vehicles -- the densest case.

Each factory function returns a freshly initialized environment so we can reset state between tests without side-effects.

In [ ]:
SAMPLING_RES = 10


def create_environment_1():
    return CalderaEnv(
        id='Environment 1',
        dim_x=100, dim_y=100,
        pit_params=DEFAULT_PIT_PARAMS, pit_weights=DEFAULT_PIT_WEIGHTS,
        sampling_res=SAMPLING_RES, movement_size=SAMPLING_RES,
        initial_position=(10, 40),
        energy_per_move=5, initial_energy=100,
        end_episode_on_collision=True,
    )


def create_environment_2():
    rng = np.random.default_rng(seed=0)
    depth_map = rng.uniform(low=-100, high=100, size=(20, 20))
    return CalderaEnv(
        id='Environment 2',
        external_depth_map=depth_map,
        sampling_res=SAMPLING_RES, movement_size=SAMPLING_RES,
        initial_position=(0, 0),
        energy_per_move=1, initial_energy=30,
        end_episode_on_collision=True,
    )


def create_environment_3():
    other_vehicles = [
        ((5, 2), SAMPLING_RES), ((30, 30), SAMPLING_RES),
        ((30, 15), SAMPLING_RES), ((80, 80), SAMPLING_RES),
        ((30, 70), SAMPLING_RES), ((60, 15), SAMPLING_RES),
        ((10, 15), SAMPLING_RES),
    ]
    return CalderaEnv(
        id='Environment 3',
        dim_x=100, dim_y=100,
        pit_params=DEFAULT_PIT_PARAMS, pit_weights=DEFAULT_PIT_WEIGHTS,
        sampling_res=SAMPLING_RES, movement_size=SAMPLING_RES,
        initial_position=(10, 40),
        energy_per_move=5, initial_energy=100,
        end_episode_on_collision=True,
        other_vehicles=other_vehicles,
    )


def create_environment_4():
    other_vehicles = [
        ((5, 2), SAMPLING_RES), ((30, 30), SAMPLING_RES),
        ((30, 15), SAMPLING_RES), ((80, 80), SAMPLING_RES),
        ((30, 70), SAMPLING_RES), ((60, 15), SAMPLING_RES),
        ((10, 15), SAMPLING_RES), ((20, 20), SAMPLING_RES),
        ((40, 40), SAMPLING_RES), ((50, 50), SAMPLING_RES),
        ((70, 70), SAMPLING_RES), ((90, 90), SAMPLING_RES),
        ((20, 30), SAMPLING_RES), ((30, 20), SAMPLING_RES),
        ((40, 50), SAMPLING_RES), ((50, 40), SAMPLING_RES),
        ((60, 70), SAMPLING_RES), ((70, 60), SAMPLING_RES),
        ((80, 90), SAMPLING_RES), ((90, 80), SAMPLING_RES),
    ]
    return CalderaEnv(
        id='Environment 4',
        dim_x=100, dim_y=100,
        pit_params=DEFAULT_PIT_PARAMS, pit_weights=DEFAULT_PIT_WEIGHTS,
        sampling_res=SAMPLING_RES, movement_size=SAMPLING_RES,
        initial_position=(10, 40),
        energy_per_move=5, initial_energy=100,
        end_episode_on_collision=True,
        other_vehicles=other_vehicles,
    )


ENVIRONMENT_FACTORIES = [
    create_environment_1,
    create_environment_2,
    create_environment_3,
    create_environment_4,
]

Quick visual sanity check: render the four reference environments side by side. The black squares are obstacle vehicles; the dot is the agent's starting position; deeper terrain shows up in darker color.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, factory in zip(axes.flat, ENVIRONMENT_FACTORIES):
    env = factory()
    frame = render_env_frame(env, show_grid_lines=True)
    ax.imshow(frame)
    ax.set_title(env.id)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Building `task_info`

Recall that callbacks in `to_implement.py` never get access to the environment object. The three explorers pass a `task_info` dict instead -- a static description of the task that includes only the information the agent is allowed to use. The helpers below construct an appropriate `task_info` for each explorer type.

In [ ]:
def al_task_info(env):
    return {
        'grid_dimensions': (env.dim_x, env.dim_y),
        'sampling_res': env.sampling_res,
        'initial_position': tuple(env.initial_position),
        'initial_energy': env.initial_energy,
    }


def planner_task_info(env):
    return {
        'grid_dimensions': (env.dim_x, env.dim_y),
        'pit_params': env.pit_params,
        'pit_weights': env.pit_weights,
        'external_depth_map': env.external_depth_map,
        'sampling_res': env.sampling_res,
        'initial_position': tuple(env.initial_position),
        'movement_size': env.movement_size,
        'initial_energy': env.initial_energy,
        'energy_per_move': env.energy_per_move,
        'energy_per_sample': env.energy_per_sample,
        'energy_per_no_op': env.energy_per_no_op,
        'other_vehicles': env.other_vehicles,
        'end_episode_on_collision': env.end_episode_on_collision,
    }


def rl_task_info(env):
    return {
        'grid_dimensions': (env.dim_x, env.dim_y),
        'sampling_res': env.sampling_res,
        'initial_position': tuple(env.initial_position),
        'movement_size': env.movement_size,
        'initial_energy': env.initial_energy,
        'energy_per_move': env.energy_per_move,
        'energy_per_sample': env.energy_per_sample,
        'energy_per_no_op': env.energy_per_no_op,
        'end_episode_on_collision': env.end_episode_on_collision,
        'action_names': ACTION_NAMES,
    }

# Codebase

The codebase we provide separates the environment, agent wrappers, helper data structures, and the functions you will implement. 

- `caldera_env.py` defines the Gymnasium-style Caldera environment. 
- `explorer.py` defines the three explorer interfaces used in this assignment: `ALExplorer`, `PlanningExplorer`, and `RLExplorer`.
- `agent.py` defines the shared runner and the `step_info` / `episode_info` data schemas.
- `value_and_policy.py` provides simple `Policy` and `QTable` classes.
- `utils.py` contains shared helper functions for positions, bounds checking, paths, and terrain generation.

⚠️ The only code file that you will submit and that we will examine will be `to_implement.py`. Thus, your work should focus on this file, where you will complete the missing elements of the different agents.


# Task 1: ALExplorer - the active learning explorer


Your first agent will be an active-learning explorer. Unlike the planning and reinforcement-learning agents, this agent is not constrained by physical movement dynamics: it can choose any valid sampling-grid location and sample there directly.

The goal is to find the deepest point in the Caldera while using a limited number of samples. A good active-learning strategy should not sample locations uniformly at random. Instead, it should prioritize locations that are likely to improve the current best observation, are still unexplored, or are otherwise informative.

For this task, you will implement two functions.

The first function selects a candidate set of sampling positions from the environment:

```python
def select_candidate_queries(
    obs: dict,
    info: dict,
    candidate_set_size: int,
    task_info: dict,
) -> list[Position]:
    ...
```


This function should return a list of candidate Positions. The active-learning explorer will score these candidates and choose the best one.

The second function assigns a score to a single candidate position:



```python
def get_query_score(
    position: Position,
    obs: dict,
    info: dict,
    task_info: dict,
) -> float:
    ...
```


Note that each call to these functions should be efficient. Your agent will call them repeatedly during an episode, so you should balance informativeness with runtime cost. A good solution chooses a useful candidate set and scores those candidates quickly.


📌 **Persistent state across calls.** Each of these callbacks is invoked once per environment step, but the explorer does **not** pass any per-instance state to them — only `obs`, `info`, `candidate_set_size`, and `task_info`. If your strategy needs to remember information across calls (for example, the active learner typically needs the history of positions it has already sampled and their values), use the module-level dictionary `to_implement.g_task_info` as persistent scratch space. Anything you write into `g_task_info` will still be there on the next call. You are responsible for detecting and resetting your state at the start of a new episode (for instance, by checking that the current position equals `task_info["initial_position"]` with full `task_info["initial_energy"]`).

The task_info component will include the following:

```python
task_info = {
    "grid_dimensions": (env.dim_x, env.dim_y),
    "sampling_res": env.sampling_res,
    "initial_position": tuple(env.initial_position),
    "initial_energy": env.initial_energy,
}
```


## Testing your Task 1 implementation

### Sanity checks

These checks validate the contract of your two callbacks: candidates are valid sampling-grid `Position`s, scores are scalar floats, the budget is respected, and the explorer's `samples_collected` counter matches the fresh samples actually taken.

In [ ]:
test = unittest.TestCase()
np.random.seed(0)

env = create_environment_1()
task_info = al_task_info(env)

# 1) candidates are valid grid positions
candidates = ti.select_candidate_queries(
    obs={'position': np.array(env.initial_position), 'value': np.array(np.inf),
         'sampled_before': 0, 'energy': env.initial_energy,
         'max_value_observed': np.array(-np.inf),
         'min_value_observed': np.array(np.inf)},
    info={},
    candidate_set_size=5,
    task_info=task_info,
)
test.assertEqual(len(candidates), 5)
for c in candidates:
    test.assertEqual(len(c), 2)
    test.assertTrue(0 <= c[0] <= env.dim_x and 0 <= c[1] <= env.dim_y)
    test.assertEqual(c[0] % env.sampling_res, 0)
    test.assertEqual(c[1] % env.sampling_res, 0)

# 2) scoring returns a real float
score = ti.get_query_score(
    position=candidates[0], obs={}, info={}, task_info=task_info,
)
test.assertIsInstance(float(score), float)

# 3) sample budget is respected
MAX_SAMPLES = 15
al = ALExplorer(
    env=env, task_info=task_info,
    max_samples=MAX_SAMPLES, candidate_set_size=5,
    select_candidate_positions_fn=ti.select_candidate_queries,
    get_query_score_fn=ti.get_query_score,
)
al.run_episode()
test.assertLessEqual(al.samples_collected, MAX_SAMPLES)
test.assertGreater(al.samples_collected, 0)

# 4) best_value_observed is set and actually came from the depth field
pos, val = al.get_best_value()
test.assertIsNotNone(val)
test.assertIsNotNone(pos)
expected = env._get_value(pos)
test.assertAlmostEqual(val, expected, places=3)

print(f'AL sanity checks passed. samples_collected={al.samples_collected}, '
      f'best={val:.1f} at {pos}')

### Visual: best-so-far over samples, and sample locations on the map

The left panel shows how the running best (deepest) value improves as more samples are taken. The right panel overlays each sampled location on the depth map, colored by sampled value -- darker dots are deeper points.

In [ ]:
np.random.seed(0)
env = create_environment_1()
al = ALExplorer(
    env=env, task_info=al_task_info(env),
    max_samples=20, candidate_set_size=5,
    select_candidate_positions_fn=ti.select_candidate_queries,
    get_query_score_fn=ti.get_query_score,
)
episode_info = al.run_episode()

fresh_samples_for_plot = [
    {
        'position': tuple(map(int, step['next_obs']['position'])),
        'value': float(np.asarray(step['next_obs']['value']).item()),
    }
    for step in episode_info['trajectory']
    if step['action'] == SAMPLE
    and int(step['next_obs'].get('sampled_before', 1)) == 0
]
assert len(fresh_samples_for_plot) > 0
values = np.array([entry['value'] for entry in fresh_samples_for_plot])
running_best = np.minimum.accumulate(values)
positions = np.array([entry['position'] for entry in fresh_samples_for_plot])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(np.arange(1, len(running_best) + 1), running_best,
             marker='o', color='C0')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Best (deepest) value so far')
axes[0].set_title('Active learner -- best so far')
axes[0].grid(alpha=0.3)

x_grid = np.arange(0, env.dim_x + 1)
y_grid = np.arange(0, env.dim_y + 1)
axes[1].contourf(x_grid, y_grid, env.depth_map, levels=20, cmap='viridis')
scatter = axes[1].scatter(positions[:, 0], positions[:, 1],
                          c=values, cmap='inferno', edgecolor='white', s=80)
axes[1].set_title('Active learner -- sample locations')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
plt.colorbar(scatter, ax=axes[1], label='Sampled value')
plt.tight_layout()
plt.show()


### Video: the active learner in action

The AL explorer **teleports** to each chosen sampling-grid cell, samples it, and moves on &mdash; it is not constrained by the movement dynamics that the planning and RL agents have to obey. As a result, the visualized "path" between consecutive samples is **not a physical trajectory**: each segment is a straight jump from one sampling cell to the next, and intermediate cells along the line are never visited or sampled.

In [ ]:
np.random.seed(0)
env = create_environment_1()
al = ALExplorer(
    env=env, task_info=al_task_info(env),
    max_samples=12, candidate_set_size=5,
    select_candidate_positions_fn=ti.select_candidate_queries,
    get_query_score_fn=ti.get_query_score,
)
frames, _ = rollout_with_frames(al)
media.show_video(frames, fps=2)

# Task 2: PlanningExplorer - the look-ahead explorer


Your second agent will be a planning-based explorer. Unlike the active-learning explorer, this agent is constrained by the environment dynamics: it must move through the grid using the available movement actions, spend energy on movement and sampling, and avoid invalid moves.

The goal is still to find the deepest point in the Caldera, but now the agent must reason about how to reach informative locations under limited resources.

In this task, you will implement a look-ahead planning strategy for `PlanningExplorer`.

A planning-based agent should consider possible future action sequences before choosing its next action. For example, it may evaluate whether it is worth spending several moves to reach a promising unsampled location, or whether it should sample nearby locations before traveling farther away.


Before execution, the planner is given a limited time slot to preprocess its decisions. This is the planning phase, which is based on the task information that is provided. 

You will implement two functions.
The first is called once during initialization. 
Note that it receives a pointer to the agent's policy, and can update it. 

```python
def init_policy_fn(
    policy: Policy,
    resources,
    objective,
    task_info: dict,
    state_key_fn: Callable,
) -> None:
    ...
```


The second function is used to update the policy. It is called at every step (when step_update is called)

```python
def update_policy_fn(
    policy: Policy,
    resources,
    objective,
    task_info: dict,
    state_key_fn: Callable,
    step_info: dict,
) -> None:
    ...
```


Here, we will provide the following information: 

```python
task_info = {
    "grid_dimensions": (env.dim_x, env.dim_y),
    "pit_params": env.pit_params,
    "pit_weights": env.pit_weights,
    "external_depth_map": env.external_depth_map,
    "sampling_res": env.sampling_res,
    "initial_position": tuple(env.initial_position),
    "movement_size": env.movement_size,
    "initial_energy": env.initial_energy,
    "energy_per_move": env.energy_per_move,
    "energy_per_sample": env.energy_per_sample,
    "energy_per_no_op": env.energy_per_no_op,
    "other_vehicles": env.other_vehicles,
    "end_episode_on_collision": env.end_episode_on_collision,
}
```


## Testing your Task 2 implementation

> ⚠️ **Important &mdash; 5-minute planning cap.** During internal grading, `init_policy_fn` is capped at **5 minutes of wall-clock time per environment**. Solutions that exceed this limit will be killed and receive no credit for that environment. The reference implementation runs in well under a second; anything heavier than a single BFS / value-iteration sweep over the sampling grid is almost certainly too much. `update_policy_fn` runs once per environment step and is bound only by the episode-length cap, so it is the appropriate place for any per-step refinement.

### Sanity checks

These checks validate that your planner installs a non-trivial policy, that the planned actions are legal, and that running the planned policy in the environment beats a do-nothing baseline.

In [ ]:
test = unittest.TestCase()
np.random.seed(0)

env = create_environment_3()  # with obstacles
task_info = planner_task_info(env)
planner = PlanningExplorer(
    env=env, task_info=task_info,
    init_policy_fn=ti.init_policy_fn,
    update_policy_fn=ti.update_policy_fn,
)

# 1) the policy is non-empty after init_policy_fn
test.assertGreater(len(planner.policy.actions), 0)

# 2) the policy entry for the initial position is a legal action
init_key = str(tuple(map(int, env.initial_position)))
test.assertIn(init_key, planner.policy)
test.assertIn(planner.policy[init_key],
              (MOVE_NORTH, MOVE_SOUTH, MOVE_EAST, MOVE_WEST, SAMPLE, NO_OP))

# 3) running the planned policy beats a do-nothing baseline
episode_info = planner.run_episode()
_, planner_best = planner.get_best_value()
test.assertIsNotNone(planner_best)
test.assertGreater(episode_info['num_steps'], 0)

print(f'Planning sanity checks passed. policy size = {len(planner.policy.actions)}, '
      f'best value = {planner_best:.1f}')

### Visual: planned path overlay

We extract the sequence of positions visited during the planned rollout and overlay it on the depth map. Black squares are obstacles; the white line is the executed trajectory; the red star marks the cell where the planner decided to sample.

In [ ]:
np.random.seed(0)
env = create_environment_3()
planner = PlanningExplorer(
    env=env, task_info=planner_task_info(env),
    init_policy_fn=ti.init_policy_fn,
    update_policy_fn=ti.update_policy_fn,
)
episode_info = planner.run_episode()
sample_pos = next(
    (tuple(int(c) for c in step['next_obs']['position'])
     for step in episode_info['trajectory'] if step['action'] == SAMPLE),
    None,
)

fig, ax = env.visualize(show_grid_lines=True, show_agent_path=True)
if sample_pos is not None:
    ax.scatter(sample_pos[0], sample_pos[1], marker='*',
               s=240, color='red', edgecolor='white', zorder=12,
               label='SAMPLE')
    ax.legend(loc='lower right')
ax.set_title(f"Planner trajectory on {env.id}")
plt.show()
print('best value found by planner:', planner.get_best_value())

### Video: the planner in action

Watch the planner move step-by-step toward the deepest reachable cell, then sample it. With the BFS reference solution, this is a shortest path through the obstacle field.

In [ ]:
np.random.seed(0)
env = create_environment_3()
planner = PlanningExplorer(
    env=env, task_info=planner_task_info(env),
    init_policy_fn=ti.init_policy_fn,
    update_policy_fn=ti.update_policy_fn,
)
frames, _ = rollout_with_frames(planner)
media.show_video(frames, fps=3)

# Task 3: RLExplorer - the RL explorer


Your third agent will be a reinforcement-learning explorer. In this task, we will use a simple Q-learning agent that learns action values from experience while interacting with the environment.

Unlike the planning explorer, the RL explorer does not rely on an explicit look-ahead model of future trajectories. Instead, it gradually learns which actions tend to lead to better outcomes based on the rewards it receives during episodes.

For this assignment, we will keep the RL setup intentionally simple and focus on the basic Q-learning loop: selecting actions, observing rewards, and updating action-value estimates. In the next assignment, we will extend this approach and explore more powerful RL methods and representations.

The RL explorer also needs an exploration schedule. You will define a global hyperparameter dictionary named `LEARNING_PARAMS` in `to_implement.py`, and implement `epsilon_update_fn(...)`. The agent calls your `epsilon_update_fn` automatically before each training action, so the notebook should not manually update `rl.epsilon`. You should tune `LEARNING_PARAMS` and the update rule yourself.


Here you will implement five functions and tune one global dictionary. The first is `state_key_fn`, which converts an observation and info dictionary into the state key used by the policy and Q-table. A good state representation should include the information needed to distinguish decisions that should have different values.

The second is `epsilon_update_fn`, which returns the epsilon value used for the next epsilon-greedy action. This function receives the current epsilon, episode and step counters, and `LEARNING_PARAMS`, and is called automatically by `RLExplorer` during training.

The third is `step_update_fn`, which gives you access to the agent's policy and Q-table after each environment step, i.e., the function is responsible for the per-step learning part of Q-learning.


```python
def state_key_fn(obs: dict, info: dict) -> str:
    ...

def epsilon_update_fn(
    current_epsilon: float,
    episode_index: int,
    step_index: int,
    total_steps: int,
    learning_params: dict,
) -> float:
    ...

def step_update_fn(policy: Policy, q_table: QTable, resources, objective, task_info, state_key_fn, step_info):
    ...
```

The function receives the mutable `policy` and `q_table`, so it does not need to return a new object. `resources`, `objective`, and `task_info` provide the task context, while `state_key_fn` converts observations into the state keys used by the policy and Q-table. The `step_info` argument contains the information from the most recent transition: the previous observation, selected action, reward, new observation, termination flags, and the `info` dictionary returned by the environment.


The fourth function to implement is `episode_update_fn`, which is called once at the end of each episode with `episode_info`. This is where Monte-Carlo-style updates can use the full trajectory if your design needs them.


📌 **`step_update_fn` vs `episode_update_fn`.** These two callbacks correspond to the two standard styles of value-based RL update and they receive different data structures:

- `step_update_fn` is called **after every environment transition** during training, with a `step_info` dict containing only that one transition. The keys are `obs`, `action`, `reward`, `next_obs`, `terminated`, `truncated`, and `info` (which may contain an `action_mask`). Use it for **TD-style** updates such as the Q-learning rule $Q(s,a) \leftarrow Q(s,a) + \alpha\,(r + \gamma \max_{a'} Q(s', a') - Q(s, a))$ that only need the most recent transition.
- `episode_update_fn` is called **once at the end of each training episode**, with an `episode_info` dict that contains the **full trajectory** (`episode_info["trajectory"]` is a list of all `step_info` dicts, in order), the `total_reward`, and the `num_steps`. Use it for **Monte-Carlo-style** updates that need the whole return from each state, or any post-episode bookkeeping.

You may implement only one of them (return `None` from the other) or use both together — the framework will dispatch to whichever you fill in.

```python
def episode_update_fn(
    policy: Policy,
    q_table: QTable,
    resources,
    objective,
    task_info: dict,
    state_key_fn: Callable,
    episode_info: dict,
) -> None:
    ...
```


The fifth function is the reward function used by the environment. The reward function defines what the RL agent is trying to learn: after each action, the environment calls this function and uses the returned scalar value as the reward for that transition.

`reward_function` has signature `(action: str, obs: dict) -> float` and receives the post-action observation, which exposes the following fields you can use to shape the reward:

- `position`: the agent's `(x, y)` after the action,
- `energy`: the remaining energy budget,
- `sampled_before`: `1` if the cell at the new position had already been sampled in this episode, `0` otherwise,
- `value`: the sampled depth value at the new position when the action just produced a fresh sample, otherwise the environment's default sentinel (an infinite value),
- `max_value_observed`, `min_value_observed`: the running max / min of all sampled values so far in this episode (the running deepest value lives in `min_value_observed`, since "deeper" is "more negative").

The reward function does **not** receive a reference to the environment object, so any shaping logic has to be expressible from `action` and these observation fields alone.

```python
LEARNING_PARAMS = {
    'learning_rate': ...,
    'discount_factor': ...,
    'epsilon_start': ...,
    'epsilon_min': ...,
    'epsilon_decay': ...,
}

def reward_function(action: str, obs: dict) -> float:
    ...
```

`LEARNING_PARAMS` is part of your solution and **you need to tune it**: the values above are hyperparameters, not fixed constants imposed by the framework. A few things to be aware of:

- The framework reads `epsilon_start` directly from `LEARNING_PARAMS` to initialize `RLExplorer.epsilon`, so this key must be present and numeric.
- Every other key listed (`learning_rate`, `discount_factor`, `epsilon_min`, `epsilon_decay`) is consumed only by your own callbacks (`epsilon_update_fn`, `step_update_fn`, `episode_update_fn`). If your callbacks don't read a key, you can leave it unset.
- You are free to **add new keys** to `LEARNING_PARAMS` for any custom hyperparameters your solution needs. The framework forwards the whole dictionary to your callbacks unchanged.

## Testing your Task 3 implementation

### Sanity checks: reward function, epsilon schedule, and TD update

First we check that `reward_function` returns the documented values, that `epsilon_update_fn` returns a valid exploration probability, and that a single TD update moves the Q-table toward the TD target.


In [ ]:
test = unittest.TestCase()

# 1) reward_function returns the documented values
r_move = ti.reward_function(MOVE_NORTH, {
    'sampled_before': 0,
    'value': np.array(0.0),
    'max_value_observed': np.array(-np.inf),
    'min_value_observed': np.array(np.inf),
})
r_repeat = ti.reward_function(SAMPLE, {
    'sampled_before': 1,
    'value': np.array(0.0),
    'max_value_observed': np.array(0.0),
    'min_value_observed': np.array(0.0),
})
r_fresh = ti.reward_function(SAMPLE, {
    'sampled_before': 0,
    'value': np.array(-50.0),
    'max_value_observed': np.array(-50.0),
    'min_value_observed': np.array(-50.0),
})
test.assertLess(r_move, 0)
test.assertLess(r_repeat, r_move)
test.assertGreater(r_fresh, r_repeat)


# 2) epsilon_update_fn returns a valid exploration probability and decays over episodes
eps0 = ti.epsilon_update_fn(
    current_epsilon=ti.LEARNING_PARAMS['epsilon_start'],
    episode_index=0,
    step_index=0,
    total_steps=0,
    learning_params=ti.LEARNING_PARAMS,
)
eps_later = ti.epsilon_update_fn(
    current_epsilon=eps0,
    episode_index=1000,
    step_index=0,
    total_steps=1000,
    learning_params=ti.LEARNING_PARAMS,
)
test.assertGreaterEqual(eps0, eps_later)
test.assertGreaterEqual(eps_later, 0.0)
test.assertLessEqual(eps0, 1.0)

# 3) step_update_fn actually shrinks the gap between Q and the TD target
from value_and_policy import Policy, QTable
policy = Policy(); q_table = QTable()
step = {
    'obs': {'position': np.array([10, 40]), 'sampled_before': 0},
    'next_obs': {'position': np.array([20, 40]), 'sampled_before': 0},
    'action': MOVE_EAST,
    'reward': 1.0,
    'terminated': False, 'truncated': False, 'info': {},
}
before = q_table.get_value(ti.state_key_fn(step['obs'], {}), MOVE_EAST)
ti.step_update_fn(
    policy, q_table, None, OBJECTIVES['deepest'],
    {'action_names': ACTION_NAMES}, ti.state_key_fn, step,
)
after = q_table.get_value(ti.state_key_fn(step['obs'], {}), MOVE_EAST)
test.assertGreater(after, before)

print(f'RL sanity checks passed. r_move={r_move:.2f}, r_repeat={r_repeat:.2f}, '
      f'r_fresh={r_fresh:.2f}; epsilon {eps0:.3f} -> {eps_later:.3f}; '
      f'Q before/after = {before:.3f} -> {after:.3f}')

### Training curve

Train a Q-learning agent on Environment 1 with an epsilon schedule: start with substantial exploration, then decay to a mostly-greedy policy. With the reference shaping (**+1 for sampling a fresh cell at or beyond the running deepest value**, **decreasing toward 0 as the gap to the running deepest value grows**, **-0.5 for sampling an already-sampled cell**, and **-0.1 for movement / NO_OP**), the smoothed reward and the best sampled value should improve as the agent learns to navigate toward the deeper regions of the map rather than just chase any unsampled cell.


In [ ]:
np.random.seed(0)

env = create_environment_1()
env.set_reward_function(ti.reward_function)

N_EPISODES = 5_000

rl = RLExplorer(
    env=env, task_info=rl_task_info(env),
    learning_params=ti.LEARNING_PARAMS,
    state_key_fn=ti.state_key_fn,
    epsilon_update_fn=ti.epsilon_update_fn,
    step_update_fn=ti.step_update_fn,
    episode_update_fn=ti.episode_update_fn,
)

print('RL learning parameters:', ti.LEARNING_PARAMS)

rewards_per_episode = []
episode_best_values = []
epsilon_values = []
for _ in tqdm(range(N_EPISODES)):
    info = rl.run_episode()  # RLExplorer calls ti.epsilon_update_fn automatically
    rewards_per_episode.append(info['total_reward'])
    epsilon_values.append(rl.epsilon)

    fresh_sample_values = [
        float(np.asarray(step['next_obs']['value']).item())
        for step in info['trajectory']
        if step['action'] == SAMPLE
        and int(step['next_obs'].get('sampled_before', 1)) == 0
    ]
    episode_best_values.append(min(fresh_sample_values) if fresh_sample_values else np.nan)

rewards = np.asarray(rewards_per_episode)
episode_best = np.asarray(episode_best_values, dtype=float)
window = 100
smoothed = np.convolve(rewards, np.ones(window) / window, mode='valid')
cumulative_best = np.minimum.accumulate(
    np.where(np.isfinite(episode_best), episode_best, np.inf)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(rewards, color='C0', alpha=0.25, label='Episode reward')
axes[0].plot(np.arange(window - 1, N_EPISODES), smoothed, color='C0',
             linewidth=2, label=f'{window}-episode moving avg')
axes[0].set_xlabel('Training episode')
axes[0].set_ylabel('Total reward')
axes[0].set_title('RL training reward')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(cumulative_best, color='C1', linewidth=2)
axes[1].set_xlabel('Training episode')
axes[1].set_ylabel('Best sampled value so far')
axes[1].set_title('RL best value discovered during training')
axes[1].grid(alpha=0.3)

axes[2].plot(epsilon_values, color='C2', linewidth=2)
axes[2].set_xlabel('Training episode')
axes[2].set_ylabel('epsilon')
axes[2].set_title('Automatic epsilon schedule')
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'q-table size after training: {len(rl.q_table.values)} entries')
print(f'final epsilon: {rl.epsilon:.3f}; best value found during training: {rl.get_best_value()[1]:.1f}')

### Q-value heatmap

For each cell on the sampling grid, look up the maximum Q-value over all actions. Cells the agent considers 'valuable to be at' should light up; regions where the Q-table is still at the default value of 0 mean the agent never visited them during training.

In [ ]:
grid_x = np.arange(0, env.dim_x + 1, env.sampling_res)
grid_y = np.arange(0, env.dim_y + 1, env.sampling_res)
q_heatmap = np.zeros((len(grid_y), len(grid_x)))
for i, y in enumerate(grid_y):
    for j, x in enumerate(grid_x):
        key = ti.state_key_fn({'position': np.array([x, y]), 'sampled_before': 0}, {})
        q_heatmap[i, j] = max(
            rl.q_table.get_value(key, a) for a in ACTION_NAMES
        )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(env.depth_map, origin='lower', cmap='viridis')
axes[0].set_title(f'{env.id} depth map')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
im = axes[1].imshow(q_heatmap, origin='lower', cmap='magma',
                    extent=(grid_x[0], grid_x[-1], grid_y[0], grid_y[-1]))
axes[1].set_title('max_a Q(s=position|sampled_before=0, a)')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
plt.colorbar(im, ax=axes[1], label='Q value')
plt.tight_layout()
plt.show()

### Video: trained RL agent acting greedily

Switch the agent to evaluation mode (`agent.eval()`) so it picks the Q-greedy action at every step (no epsilon exploration, no learning updates) and watch a single rollout.

In [ ]:
rl.eval()
np.random.seed(0)
target_value = rl.get_best_value()[1]
frames, eval_info = rollout_with_frames(
    rl,
    stop_sample_value_at_most=target_value + 1e-6,
)
eval_sample_values = [
    float(np.asarray(step['next_obs']['value']).item())
    for step in eval_info['trajectory']
    if step['action'] == SAMPLE
    and int(step['next_obs'].get('sampled_before', 1)) == 0
]
eval_best = min(eval_sample_values) if eval_sample_values else None
final_position = tuple(map(int, eval_info['trajectory'][-1]['next_obs']['position']))
print(f'Eval rollout: {eval_info["num_steps"]} steps, '
      f'total reward = {eval_info["total_reward"]:.2f}, '
      f'eval best value = {eval_best:.1f}, final position = {final_position}')
media.show_video(frames, fps=3)


# Evaluation

Evaluation will be performed on a different environment and with various resource settings. The expected evaluation flow is the same as the end-to-end comparison below: environments are created, each explorer is run with the functions from `to_implement.py`, and the resulting performance is compared. This means your solution should avoid hard-coding assumptions about a single map, initial position, or energy budget, and should instead use the observations, `task_info`, and allowed helper functions provided to the agent.


A strong solution is one that behaves reasonably across multiple environments, not just one that performs well on the visible examples. As you test your agents, ask whether each decision rule would still make sense if the terrain changed, if energy became scarce, or if the most valuable region was far from the starting point.

## End-to-end evaluation

Finally, run all three explorers on all four environments and compare their best-value-observed side by side. This is roughly the comparison the grader will run on a held-out environment, so a strong solution should look robust here too -- not just on Environment 1.

In [ ]:
AL_SAMPLES = 15
RL_EPISODES = 3_000
EXPLORERS = ['AL', 'Plan', 'RL']
results = {explorer: [] for explorer in EXPLORERS}
env_names = []

for factory in ENVIRONMENT_FACTORIES:
    env_names.append(factory().id)

    # active learner
    np.random.seed(0)
    env = factory()
    al = ALExplorer(
        env=env, task_info=al_task_info(env),
        max_samples=AL_SAMPLES, candidate_set_size=5,
        select_candidate_positions_fn=ti.select_candidate_queries,
        get_query_score_fn=ti.get_query_score,
    )
    al.run_episode()
    results['AL'].append(al.get_best_value()[1])

    # planner
    np.random.seed(0)
    env = factory()
    planner = PlanningExplorer(
        env=env, task_info=planner_task_info(env),
        init_policy_fn=ti.init_policy_fn,
        update_policy_fn=ti.update_policy_fn,
    )
    planner.run_episode()
    results['Plan'].append(planner.get_best_value()[1])

    # RL
    np.random.seed(0)
    env = factory()
    env.set_reward_function(ti.reward_function)
    rl = RLExplorer(
        env=env, task_info=rl_task_info(env),
        learning_params=ti.LEARNING_PARAMS,
        state_key_fn=ti.state_key_fn,
        epsilon_update_fn=ti.epsilon_update_fn,
        step_update_fn=ti.step_update_fn,
        episode_update_fn=ti.episode_update_fn,
    )
    rl.run_episodes(num_episodes=RL_EPISODES)
    rl.eval()
    rl.run_episode()
    results['RL'].append(rl.get_best_value()[1])

# Print as a small text table.
header = f"{'env':<16}" + ''.join(f'{e:>12}' for e in EXPLORERS)
print(header)
print('-' * len(header))
for i, name in enumerate(env_names):
    row = f"{name:<16}" + ''.join(
        f'{results[e][i]:>12.1f}' if results[e][i] is not None else f"{'N/A':>12}"
        for e in EXPLORERS
    )
    print(row)

In [ ]:
# Environment 2 uses an external depth-map with values around [-100, 100],
# while the Gaussian-pit environments are on a much larger negative scale.
# Plot Environment 2 separately so its bars remain readable.
env2_index = env_names.index('Environment 2')
large_scale_indices = [i for i in range(len(env_names)) if i != env2_index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), width_ratios=[3, 1])
bar_width = 0.25

for ax, indices, title in [
    (axes[0], large_scale_indices, 'Gaussian-pit environments'),
    (axes[1], [env2_index], 'External-map environment'),
]:
    positions = np.arange(len(indices))
    for explorer_index, explorer in enumerate(EXPLORERS):
        values = [
            results[explorer][env_index]
            if results[explorer][env_index] is not None else 0.0
            for env_index in indices
        ]
        ax.bar(
            positions + explorer_index * bar_width,
            values,
            width=bar_width,
            edgecolor='black',
            label=explorer,
        )
    ax.set_xticks(positions + bar_width)
    ax.set_xticklabels([env_names[i] for i in indices], rotation=0)
    ax.axhline(0, color='gray', linewidth=1)
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel('Best (deepest) value observed')
axes[1].set_ylabel('Best value observed (Env 2 scale)')
axes[0].legend()
fig.suptitle('Best value observed across explorers and environments')
plt.tight_layout()
plt.show()


## Task 4: Dry Questions


### Question 4: Formalizing the explorer algorithms

In tasks 1-3 you implemented three different decision-making styles for the same Caldera exploration task:

1. an active-learning explorer that chooses sampling locations,
2. a planning explorer that commits to a policy before acting
3. a tabular reinforcement-learning explorer that updates a Q-table using TD-error (Q-learning).

In this question, you will formalize these algorithms. You should describe your own implementation, but your notation should make clear what state variables, objective, action set, and update rules are being used.

Use the following notation unless your implementation requires an extension:

- The map coordinates are
  $$
  V=\{0,\ldots,d_x\}\times \{0,\ldots,d_y\}.
  $$
- The sampling grid is
  $$
  G=\{(ir,jr): ir\le d_x,\ jr\le d_y\},
  $$
  where $r$ is `sampling_res`.
- The objective is **deepest cell**, so smaller depth values are better.
- $f(g)$ is the depth value of sampling-grid cell $g\in G$.
- $p_t\in V$ is the agent position, $e_t$ is remaining energy, and $H_t\subseteq G\times\mathbb R$ is the set of sampled cells and their observed values.
- The action set is
  $$
  A=\{\texttt{MOVE\_NORTH},\texttt{MOVE\_SOUTH},\texttt{MOVE\_EAST},\texttt{MOVE\_WEST},\texttt{SAMPLE},\texttt{NO\_OP}\}.
  $$


### ❓ Question 4(a) ❓

Define a formal decision-making model for the HW2 Caldera exploration problem. Your answer should define the state, actions, transition dynamics at a high level, observations, reward/objective, and episode budget. Write your answer in `answers.py:q4a`.


In [ ]:
Markdown(answers.q4a)


### ❓ Question 4(b) ❓

Write pseudocode for your active-learning explorer. Then state at least two guarantees or properties of the algorithm, such as validity of returned candidates, runtime per decision, monotonicity of the incumbent best sample, or an anytime/sample-budget property. If you use a surrogate model such as a Gaussian process or RBF kernel, state the acquisition score you use. Write your answer in `answers.py:q4b`.


In [ ]:
Markdown(answers.q4b)


### ❓ Question 4(c) ❓

Write pseudocode for your planning explorer. Then prove that the produced policy is sound: every planned movement is legal and the final `SAMPLE` action is executable under the energy budget. Also state a completeness or optimality guarantee for the class of plans searched by your algorithm, and give its runtime in terms of the grid size and obstacle-checking cost. You may cite existing literature as proof. Write your answer in `answers.py:q4c`.


In [ ]:
Markdown(answers.q4c)


### ❓ Question 4(d) ❓

Write pseudocode for your reinforcement-learning explorer. Include the state-key map, action-selection rule, epsilon schedule, and Q-update. Then state the conditions under which tabular Q-learning is guaranteed to converge to the optimal action-value function. Also explain which of those conditions are only approximately satisfied by a finite HW2 training run. Write your answer in `answers.py:q4d`.


In [ ]:
Markdown(answers.q4d)


### ❓ Question 4(e) ❓

Compare the three methods from parts (b)-(d). Which guarantees are strongest before execution, which method is most naturally anytime, and which method can improve from repeated interaction with the environment? Write your answer in `answers.py:q4e`.


In [ ]:
Markdown(answers.q4e)


### Question 5: Structured particles for a partially observable Caldera POMDP

From the lectures, recall the following ideas:

- In a partially observable model, the agent does not know the true hidden state. It maintains a **belief**
  $$
  \beta_t(s)=P(S_t=s\mid h_t),
  $$
  where $h_t$ is the observation/action history.
- A Bayes filter updates a belief in two stages: a **prediction** step using the dynamics and a **correction** step using the observation likelihood.
- When the state space is large, a belief can be approximated by **particles**. A particle is one possible hidden state; a weighted particle set approximates a probability distribution over hidden states.

In this question we use a slightly more structured particle representation. The name for this idea is **Rao-Blackwellization**. Instead of sampling every hidden variable, each particle samples only the hard discrete part and stores an exact/simple distribution for the part that can be updated analytically.

Reference (optional): this structured-particle idea is commonly called **Rao-Blackwellized particle filtering**; see [Doucet, de Freitas, Murphy, and Russell, "Rao-Blackwellised Particle Filtering for Dynamic Bayesian Networks" (UAI 2000)](https://www.cs.ubc.ca/~murphyk/Papers/rbpf_uai00.pdf).

For this problem, focus on a single sampling-grid cell $g$. The hidden variables relevant to this cell are:

- $M_t$: a hidden discrete map/geological hypothesis. You can think of $M_t$ as a possible explanation of the local terrain and obstacles.
- $D_g\in\mathbb R$: the true depth value at cell $g$.

The agent exactly observes its own position and energy. It may also receive:

- a binary local sensor observation $Z\in\{0,1\}$, whose likelihood depends on the map hypothesis:
  $$
  P(Z=z\mid M=m),
  $$
- and, if it executes `SAMPLE` at cell $g$, a noisy depth observation
  $$
  Y=D_g+\epsilon,\qquad \epsilon\sim\mathcal N(0,\sigma_y^2).
  $$

The structured particle belief is a set
$$
\mathcal B_t=\{(m_t^i,w_t^i,\mu_t^i,v_t^i)\}_{i=1}^N,
$$
where particle $i$ means:

- the sampled discrete hypothesis is $M_t=m_t^i$,
- its normalized particle weight is $w_t^i$,
- conditioned on $M_t=m_t^i$ and the history, the depth of cell $g$ is represented as
  $$
  D_g\mid M_t=m_t^i,h_t\sim \mathcal N(\mu_t^i,v_t^i).
  $$

So each particle is not just a sampled full state. It is a sampled map hypothesis **plus a small Gaussian belief** over the depth value, conditioned on the map hypothesis and observation history.


### ❓ Question 5(a) ❓

Using the notation above, write the belief distribution represented by the structured particle set $\mathcal B_t$. Write your answer in `answers.py:q5a`.

*Hint:* The answer should look like a weighted mixture: each term fixes one value of $M$ and attaches a Gaussian distribution over $D_g$.


In [ ]:
Markdown(answers.q5a)


### ❓ Question 5(b) ❓

Write pseudocode for one update of this structured particle belief after an action and a new observation. Write your answer in `answers.py:q5b`.

Your pseudocode should follow the same shape as the Bayes filter from lecture: predict, correct, normalize. Include optional resampling using
$$
N_{\mathrm{ess}}=\frac{1}{\sum_i (w^i)^2}.
$$

*Hint:* If the map hypothesis is static, the prediction step for $m^i$ is just $m^{i,+}=m^i$. If the action is not `SAMPLE`, there is no depth-measurement correction and the Gaussian can remain unchanged.


In [ ]:
Markdown(answers.q5b)


### ❓ Question 5(c) ❓

Suppose particle $i$ currently stores
$$
D_g\mid M=m^i,h_t\sim \mathcal N(\mu_i,v_i),
$$
and the agent samples cell $g$ and observes $Y=y$, with noise variance $\sigma_y^2$. Derive the posterior mean, posterior variance, and the likelihood term for this particle. Write your answer in `answers.py:q5c`.

*Hint:* This is a one-dimensional Gaussian conditioning problem. A convenient intermediate quantity is $S_i=v_i+\sigma_y^2$.


In [ ]:
Markdown(answers.q5c)


### ❓ Question 5(d) ❓

Compute one numerical update. There are two structured particles:

| particle | map hypothesis | prior weight | prior for $D_g$ | local-sensor likelihood $P(Z=1\mid M)$ |
|---|---:|---:|---:|---:|
| A | $M=A$ | $0.6$ | $\mathcal N(-100,25)$ | $0.8$ |
| B | $M=B$ | $0.4$ | $\mathcal N(-80,16)$ | $0.2$ |

The agent executes `SAMPLE` at cell $g$ and observes
$$
Z=1,\qquad Y=-92.
$$
The sample-noise variance is $\sigma_y^2=9$.

Compute:

1. the posterior Gaussian for $D_g$ inside each particle,
2. the normalized particle weights,
3. the effective sample size $N_{\mathrm{ess}}$,
4. whether the filter should resample if the threshold is $1.5$.

Write your answer in `answers.py:q5d`.

*Hint:* The unnormalized weight has the form
$$
\tilde w_i=w_i\,P(Z=1\mid M=m_i)\,P(Y=-92\mid M=m_i,h_t).
$$


In [ ]:
Markdown(answers.q5d)


### ❓ Question 5(e) ❓

Compare this structured particle filter to an ordinary particle filter that samples both $M$ and $D_g$ explicitly. Explain why storing a Gaussian distribution for $D_g$ inside each particle can reduce variance. Write your answer in `answers.py:q5e`.

*Hint:* Think about whether a particle needs to get lucky by sampling a depth value close to the observation $y$, or whether it can integrate over all possible depth values under its Gaussian.


In [ ]:
Markdown(answers.q5e)


<!-- <img src="../images/good-luck-have-fun.jpeg" alt="Good luck, have fun" width="200" /> -->
![Good luck, have fun](../images/good-luck-have-fun.jpeg)